# 03 - OCR single video (optimized PP-OCRv5, full-video default)

This stage reads keyframes from notebook 01 and writes one OCR record per
`keyframe_id`. The production default processes every keyframe in the selected
video. Set `OCR_DEBUG_MAX_FRAMES` to a positive number only for a smoke test.

OCR keeps scene text and ticker text typed separately. Stage 06 uses scene text
normally and keeps ticker text as lower-weight evidence.

If an earlier run raised `libtorch_cuda.so: undefined symbol: ncclCommShrink`,
stop the Kaggle session and start a fresh GPU session before running this
notebook from the first cell.


In [ ]:
# Keep Kaggle's PyTorch CUDA/NCCL stack authoritative. The Paddle package
# index carries an older NCCL build that can break torch with ncclCommShrink.
import importlib.metadata as package_metadata
import subprocess
import sys

from packaging.requirements import Requirement

if "paddle" in sys.modules:
    raise RuntimeError(
        "Paddle is already loaded. Stop this Kaggle session, start a fresh GPU "
        "session, then run the updated notebook from the first cell."
    )

torch_requirements = [
    Requirement(item)
    for item in (package_metadata.requires("torch") or [])
]
torch_nccl_requirement = next(
    (
        item
        for item in torch_requirements
        if item.name.casefold() == "nvidia-nccl-cu12"
    ),
    None,
)
if torch_nccl_requirement is not None:
    try:
        installed_nccl = package_metadata.version(
            "nvidia-nccl-cu12"
        )
    except package_metadata.PackageNotFoundError:
        installed_nccl = None
    if (
        installed_nccl is None
        or installed_nccl not in torch_nccl_requirement.specifier
    ):
        torch_nccl_target = (
            f"{torch_nccl_requirement.name}"
            f"{torch_nccl_requirement.specifier}"
        )
        print(
            "Repairing PyTorch NCCL:",
            installed_nccl,
            "->",
            torch_nccl_target,
        )
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                "--force-reinstall",
                torch_nccl_target,
            ]
        )

# Load torch before Paddle so modelscope reuses the compatible NCCL library.
import torch as torch_bootstrap

print(
    "Preloaded torch:",
    torch_bootstrap.__version__,
    "CUDA:",
    torch_bootstrap.version.cuda,
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "--no-deps",
        "paddlepaddle-gpu==3.3.1",
        "--index-url",
        "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
    ]
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "paddleocr==3.7.0",
    ]
)

# Optional quality fallback. Enable only with USE_VIETOCR_FALLBACK = True.
# subprocess.check_call(
#     [sys.executable, "-m", "pip", "install", "-q", "vietocr==0.3.13"]
# )


In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
import shutil
import unicodedata
from functools import lru_cache
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import paddle
from IPython.display import display
from PIL import Image, _util as PIL_util

if not hasattr(PIL_util, "is_directory"):
    PIL_util.is_directory = (
        lambda value: isinstance(value, (str, bytes, os.PathLike)) and os.path.isdir(value)
    )

VIDEO_ID = "L21_V001"
PIPELINE_VERSION = "aicv3-ocr-optimized-v6-full-video"
OCR_MIN_CONFIDENCE = 0.45
OCR_MAX_IMAGE_SIDE = 1600
OCR_CHECKPOINT_EVERY = 20
OCR_DEBUG_MAX_FRAMES = None  # Set a positive number for a smoke test.
PREPROCESS_IMAGE = True

# False is fastest. True invokes VietOCR only for uncertain text boxes.
USE_VIETOCR_FALLBACK = False
VIETOCR_TRIGGER_CONFIDENCE = 0.65
VIETOCR_MAX_BOXES_PER_FRAME = 3
MIN_TEXT_LENGTH = 2

LOGO_ZONE_TOP_RATIO = 0.14
LOGO_ZONE_LEFT_RATIO = 0.20
LOGO_ZONE_RIGHT_RATIO = 0.20
TICKER_ZONE_CENTER_RATIO = 0.80

matches = sorted(Path("/kaggle/input").rglob("keyframes.parquet"))
if not matches:
    raise FileNotFoundError("No keyframes.parquet found under /kaggle/input")

KEYFRAMES_PATH = next(
    (path for path in matches if VIDEO_ID.lower() in path.as_posix().lower()),
    matches[0],
)
INPUT_ROOT = KEYFRAMES_PATH.parent
OUTPUT_ROOT = Path(f"/kaggle/working/03-ocr-optimized-{VIDEO_ID}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_ROOT / "ocr-checkpoint.jsonl"

keyframes_df = pd.read_parquet(KEYFRAMES_PATH)
required = {"keyframe_id", "video_id", "frame_idx", "pts_time", "frame_path"}
missing = required - set(keyframes_df.columns)
if missing:
    raise ValueError(f"keyframes.parquet missing columns: {sorted(missing)}")

keyframes_df = keyframes_df[keyframes_df.video_id.astype(str) == VIDEO_ID].copy()
keyframes_df = keyframes_df.sort_values("frame_idx").reset_index(drop=True)
if keyframes_df.empty:
    raise ValueError(f"No keyframes found for {VIDEO_ID}")

if OCR_DEBUG_MAX_FRAMES is not None and OCR_DEBUG_MAX_FRAMES < len(keyframes_df):
    sample_indices = np.linspace(
        0, len(keyframes_df) - 1, int(OCR_DEBUG_MAX_FRAMES), dtype=int
    )
    keyframes_df = keyframes_df.iloc[np.unique(sample_indices)].reset_index(drop=True)

HAS_SHOT_ID = "shot_id" in keyframes_df.columns
print("Input:", INPUT_ROOT)
print("Keyframes:", len(keyframes_df))
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Paddle:", paddle.__version__)
print("GPU build:", paddle.device.is_compiled_with_cuda())
if not paddle.device.is_compiled_with_cuda():
    raise RuntimeError("Enable a Kaggle GPU before running PaddleOCR.")
paddle.set_device("gpu:0")


In [ ]:
from paddleocr import PaddleOCR

def build_paddle_ocr():
    kwargs = {
        "lang": "vi",
        "ocr_version": "PP-OCRv5",
        "use_doc_orientation_classify": False,
        "use_doc_unwarping": False,
        "use_textline_orientation": False,
        "device": "gpu:0",
    }
    try:
        return PaddleOCR(**kwargs)
    except TypeError:
        # Compatibility path for an older PaddleOCR API in a cached Kaggle image.
        return PaddleOCR(lang="vi", use_angle_cls=False, use_gpu=True)

paddle_ocr = build_paddle_ocr()
vietocr_predictor = None

if USE_VIETOCR_FALLBACK:
    try:
        import torch
        from vietocr.tool.config import Cfg
        from vietocr.tool.predictor import Predictor

        vietocr_config = Cfg.load_config_from_name("vgg_transformer")
        vietocr_config["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"
        vietocr_config["predictor"]["beamsearch"] = True
        vietocr_predictor = Predictor(vietocr_config)
        print("VietOCR fallback ready on", vietocr_config["device"])
    except ImportError as exc:
        raise RuntimeError(
            "VietOCR fallback is enabled but vietocr is not installed. "
            "Run the optional install line in the previous cell."
        ) from exc

print("PaddleOCR ready:", type(paddle_ocr).__name__)


In [ ]:
def normalize_text(value: str) -> str:
    value = unicodedata.normalize("NFC", str(value or ""))
    return re.sub(r"\s+", " ", value).strip()

def temporal_key(text: str) -> str:
    value = unicodedata.normalize("NFD", normalize_text(text).casefold())
    value = "".join(char for char in value if unicodedata.category(char) != "Mn")
    return re.sub(r"[^a-z0-9]+", "", value)

def parse_payload(page) -> dict:
    payload = getattr(page, "json", page)
    if callable(payload):
        payload = payload()
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    return payload if isinstance(payload, dict) else {}

def first_payload_value(payload: dict, names, default):
    for name in names:
        if name in payload and payload[name] is not None:
            return payload[name]
    return default

def box_to_poly(box) -> np.ndarray:
    array = np.asarray(box, dtype="float32")
    if array.shape == (4,):
        x1, y1, x2, y2 = array.tolist()
        return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype="float32")
    points = array.reshape(-1, 2)
    if len(points) == 2:
        x1, y1 = points[0]
        x2, y2 = points[1]
        return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype="float32")
    return points[:4].astype("float32")

def jsonable_box(points):
    array = np.asarray(points, dtype="float32").reshape(-1, 2)
    return [[float(x), float(y)] for x, y in array]

def looks_reasonable_text(text: str) -> bool:
    value = normalize_text(text)
    key = temporal_key(value)
    return (
        len(value) >= MIN_TEXT_LENGTH
        and bool(re.search(r"\w", value))
        and not re.search(r"(.)\1{4,}", key)
    )

def is_static_overlay(text: str) -> bool:
    compact = re.sub(r"\s+", "", normalize_text(text)).upper()
    return bool(
        re.fullmatch(
            r"(?:HTV\w*|TV|HD|\d{1,2}:\d{2}(?::\d{2})?|LIVE|TRỰCTIẾP)",
            compact,
        )
    )

def is_valid_line_box(points: np.ndarray, width: int, height: int) -> bool:
    x1, y1 = points[:, 0].min(), points[:, 1].min()
    x2, y2 = points[:, 0].max(), points[:, 1].max()
    box_width, box_height = x2 - x1, y2 - y1
    return box_width >= 10 and box_height >= 8 and box_height <= height * 0.40

def classify_zone(points: np.ndarray, width: int, height: int, text: str) -> str:
    x_center = float((points[:, 0].min() + points[:, 0].max()) / (2 * width))
    y_center = float((points[:, 1].min() + points[:, 1].max()) / (2 * height))
    compact = re.sub(r"\s+", "", text).casefold()
    if (
        y_center < LOGO_ZONE_TOP_RATIO
        and (x_center < LOGO_ZONE_LEFT_RATIO or x_center > 1 - LOGO_ZONE_RIGHT_RATIO)
        and (len(compact) <= 12 or "htv" in compact or "online" in compact)
    ):
        return "logo"
    if y_center > TICKER_ZONE_CENTER_RATIO:
        return "ticker"
    return "scene"

def crop_axis_aligned(image: np.ndarray, points: np.ndarray, pad: int = 4) -> np.ndarray:
    x1, y1 = np.floor(points.min(axis=0)).astype(int)
    x2, y2 = np.ceil(points.max(axis=0)).astype(int)
    crop = image[max(0, y1):y2 + 1, max(0, x1):x2 + 1]
    if crop.size == 0:
        return np.empty((0, 0, 3), dtype=image.dtype)
    return cv2.copyMakeBorder(crop, pad, pad, pad, pad, cv2.BORDER_REPLICATE)

def prepare_ocr_image(path: Path) -> np.ndarray:
    image = cv2.imread(str(path))
    if image is None:
        image = cv2.cvtColor(
            np.asarray(Image.open(path).convert("RGB")),
            cv2.COLOR_RGB2BGR,
        )
    height, width = image.shape[:2]
    scale = min(1.0, OCR_MAX_IMAGE_SIDE / max(height, width))
    if scale < 0.99:
        image = cv2.resize(
            image,
            (round(width * scale), round(height * scale)),
            interpolation=cv2.INTER_AREA,
        )
    if PREPROCESS_IMAGE:
        image = cv2.convertScaleAbs(image, alpha=1.08, beta=0)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

@lru_cache(maxsize=4096)
def resolve_frame_path(frame_value: str) -> Path:
    value = Path(frame_value)
    if value.is_absolute() and value.exists():
        return value
    candidates = [
        INPUT_ROOT / value,
        INPUT_ROOT.parent / value,
        INPUT_ROOT / "frames" / value,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    basename_matches = list(INPUT_ROOT.rglob(value.name))
    if basename_matches:
        return basename_matches[0]
    raise FileNotFoundError(frame_value)

def choose_text(paddle_text: str, confidence: float, viet_text: str) -> str:
    paddle_text = normalize_text(paddle_text)
    viet_text = normalize_text(viet_text)
    if looks_reasonable_text(paddle_text) and confidence >= 0.60:
        return paddle_text
    if looks_reasonable_text(viet_text):
        return viet_text
    return paddle_text if looks_reasonable_text(paddle_text) else viet_text

def extract_detections(processed_rgb: np.ndarray, original_rgb: np.ndarray) -> list[dict]:
    pages = paddle_ocr.predict(processed_rgb)
    scale_xy = np.array(
        [
            original_rgb.shape[1] / max(processed_rgb.shape[1], 1),
            original_rgb.shape[0] / max(processed_rgb.shape[0], 1),
        ],
        dtype="float32",
    )

    raw_items = []
    for page in pages or []:
        payload = parse_payload(page)
        boxes = first_payload_value(
            payload, ("rec_polys", "dt_polys", "rec_boxes"), []
        )
        texts = first_payload_value(
            payload, ("rec_texts", "rec_text"), [""] * len(boxes)
        )
        scores = first_payload_value(
            payload, ("rec_scores", "rec_score"), [1.0] * len(boxes)
        )
        for box, text, score in zip(boxes, texts, scores):
            confidence = float(score)
            text = normalize_text(text)
            points_processed = box_to_poly(box)
            if confidence < OCR_MIN_CONFIDENCE or not text:
                continue
            if not is_valid_line_box(
                points_processed, processed_rgb.shape[1], processed_rgb.shape[0]
            ):
                continue
            if is_static_overlay(text):
                continue
            points_original = points_processed * scale_xy
            raw_items.append((points_original, text, confidence))

    raw_items.sort(key=lambda item: item[2], reverse=True)
    detections = []
    fallback_count = 0
    for points, paddle_text, confidence in raw_items:
        viet_text = ""
        if (
            vietocr_predictor is not None
            and confidence < VIETOCR_TRIGGER_CONFIDENCE
            and fallback_count < VIETOCR_MAX_BOXES_PER_FRAME
        ):
            crop = crop_axis_aligned(original_rgb, points)
            if crop.size:
                try:
                    viet_text = normalize_text(
                        vietocr_predictor.predict(Image.fromarray(crop))
                    )
                except Exception:
                    viet_text = ""
            fallback_count += 1

        final_text = choose_text(paddle_text, confidence, viet_text)
        if not final_text:
            continue
        detections.append(
            {
                "text": final_text,
                "paddle_text": paddle_text,
                "viet_text": viet_text,
                "language": "vi-en-latin",
                "confidence": confidence,
                "bbox": jsonable_box(points),
                "text_type": classify_zone(
                    points,
                    original_rgb.shape[1],
                    original_rgb.shape[0],
                    final_text,
                ),
            }
        )

    unique = {}
    for item in detections:
        key = temporal_key(item["text"])
        if key and (
            key not in unique
            or item["confidence"] > unique[key]["confidence"]
        ):
            unique[key] = item
    return sorted(
        unique.values(),
        key=lambda item: (
            min(point[1] for point in item["bbox"]),
            min(point[0] for point in item["bbox"]),
        ),
    )

print("OCR helpers ready.")


In [ ]:
# Fail before the long run if the engine or frame resolver is broken.
smoke_detections = []
smoke_error = None
for smoke_row in keyframes_df.head(5).itertuples(index=False):
    try:
        smoke_path = resolve_frame_path(str(smoke_row.frame_path))
        smoke_original = np.asarray(Image.open(smoke_path).convert("RGB"))
        smoke_processed = prepare_ocr_image(smoke_path)
        smoke_detections = extract_detections(smoke_processed, smoke_original)
        smoke_error = None
        break
    except Exception as exc:
        smoke_error = exc

if smoke_error is not None:
    raise RuntimeError(
        f"OCR smoke test failed: {type(smoke_error).__name__}: {smoke_error}"
    ) from smoke_error

print(
    "OCR smoke test passed:",
    smoke_row.keyframe_id,
    "detections=",
    len(smoke_detections),
)


In [ ]:
def write_checkpoint(records_by_id: dict):
    temporary = CHECKPOINT_PATH.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for item in sorted(
            records_by_id.values(), key=lambda value: value["frame_idx"]
        ):
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
    temporary.replace(CHECKPOINT_PATH)

completed = {}
if CHECKPOINT_PATH.exists():
    for line in CHECKPOINT_PATH.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        if item.get("pipeline_version") == PIPELINE_VERSION:
            completed[str(item["keyframe_id"])] = item
    print("Resume records:", len(completed))

for index, row in enumerate(keyframes_df.itertuples(index=False), start=1):
    keyframe_id = str(row.keyframe_id)
    previous = completed.get(keyframe_id)
    if previous is not None and not previous.get("ocr_error"):
        continue

    detections = []
    error = None
    try:
        frame_path = resolve_frame_path(str(row.frame_path))
        original_rgb = np.asarray(Image.open(frame_path).convert("RGB"))
        processed_rgb = prepare_ocr_image(frame_path)
        detections = extract_detections(processed_rgb, original_rgb)
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"

    frame_text = normalize_text(
        " ".join(
            item["text"]
            for item in detections
            if item["text_type"] != "logo"
        )
    )
    completed[keyframe_id] = {
        "keyframe_id": keyframe_id,
        "video_id": str(row.video_id),
        "frame_idx": int(row.frame_idx),
        "pts_time": float(row.pts_time),
        "shot_id": str(getattr(row, "shot_id", "")),
        "frame_path": str(row.frame_path),
        "text": frame_text,
        "raw_text": frame_text,
        "normalized_text": frame_text,
        "search_text": frame_text.casefold(),
        "detections": detections,
        "shot_text": "",
        "shot_search_text": "",
        "merged_detections": [],
        "ocr_error": error,
        "pipeline_version": PIPELINE_VERSION,
    }

    if (
        index == 1
        or index % OCR_CHECKPOINT_EVERY == 0
        or index == len(keyframes_df)
    ):
        write_checkpoint(completed)
        print(f"OCR: {index}/{len(keyframes_df)}")

records = sorted(completed.values(), key=lambda item: item["frame_idx"])
if not records:
    raise RuntimeError("No OCR records were produced.")

# Merge stable text within each shot after all available frames are processed.
groups = {}
for item in records:
    group_key = item["shot_id"] or "__video__"
    groups.setdefault(group_key, []).append(item)

for group_items in groups.values():
    merged = {}
    for frame_item in group_items:
        seen = set()
        for detection in frame_item.get("detections", []):
            if detection.get("text_type") == "logo":
                continue
            key = temporal_key(detection.get("text", ""))
            if not key or key in seen:
                continue
            seen.add(key)
            entry = merged.setdefault(
                key, {"support": 0, "best": detection}
            )
            entry["support"] += 1
            if (
                detection.get("confidence", 0.0)
                > entry["best"].get("confidence", 0.0)
            ):
                entry["best"] = detection

    merged_detections = []
    for entry in merged.values():
        detection = dict(entry["best"])
        detection["temporal_support"] = int(entry["support"])
        merged_detections.append(detection)
    merged_detections.sort(
        key=lambda item: (
            -item["temporal_support"],
            -item.get("confidence", 0.0),
        )
    )
    shot_text = normalize_text(
        " ".join(item["text"] for item in merged_detections)
    )
    for frame_item in group_items:
        frame_item["shot_text"] = shot_text
        frame_item["shot_search_text"] = shot_text.casefold()
        frame_item["merged_detections"] = merged_detections

write_checkpoint({item["keyframe_id"]: item for item in records})

ocr_df = pd.DataFrame(records).sort_values("frame_idx").reset_index(drop=True)
error_count = int(ocr_df["ocr_error"].notna().sum())
if error_count / max(len(ocr_df), 1) > 0.10:
    raise RuntimeError(
        f"OCR failed for {error_count}/{len(ocr_df)} keyframes."
    )

ocr_df.to_json(
    OUTPUT_ROOT / "ocr.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
parquet_df = ocr_df.copy()
for column in ("detections", "merged_detections"):
    parquet_df[f"{column}_json"] = parquet_df[column].map(
        lambda value: json.dumps(value, ensure_ascii=False)
    )
    parquet_df = parquet_df.drop(columns=[column])
parquet_df.to_parquet(OUTPUT_ROOT / "ocr.parquet", index=False)

all_detections = [
    detection
    for item in records
    for detection in item.get("detections", [])
]
summary = {
    "stage": "ocr",
    "pipeline_version": PIPELINE_VERSION,
    "video_id": VIDEO_ID,
    "engine": "PaddleOCR PP-OCRv5 Vietnamese/Latin",
    "vietocr_fallback_enabled": bool(USE_VIETOCR_FALLBACK),
    "keyframes_processed": int(len(ocr_df)),
    "keyframes_with_text": int((ocr_df["text"].str.len() > 0).sum()),
    "detections": int(len(all_detections)),
    "ticker_detections": int(
        sum(item.get("text_type") == "ticker" for item in all_detections)
    ),
    "errors": error_count,
    "debug_max_frames": OCR_DEBUG_MAX_FRAMES,
}
(OUTPUT_ROOT / "ocr-summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(
    ocr_df[ocr_df.text.str.len() > 0][
        ["video_id", "frame_idx", "pts_time", "text", "shot_text", "ocr_error"]
    ].head(30)
)


In [ ]:
# Archive is optional because it adds disk I/O.
MAKE_ARCHIVE = False
if MAKE_ARCHIVE:
    archive = shutil.make_archive(
        f"/kaggle/working/03-ocr-optimized-{VIDEO_ID}",
        "zip",
        root_dir=OUTPUT_ROOT,
    )
    print("Download:", archive)

del paddle_ocr
if vietocr_predictor is not None:
    del vietocr_predictor
gc.collect()
try:
    paddle.device.cuda.empty_cache()
except Exception:
    pass
print("OCR stage complete; GPU memory released.")
